In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/global-seismic-trends/data/earthquakes_clean.csv")

df.head()

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,type,year,month,day,hour,weekday,region,country_code,country,continent
0,av91018173,2021-01-30 23:57:43.470,1612139465880,51.841833,-178.581000,22.48,1.44,ml,"134 km W of Adak, Alaska",reviewed,...,earthquake,2021,1,30,23,Saturday,Alaska,US,Dutch Harbor,North America
1,nc73518690,2021-01-30 23:51:19.680,1612472529323,38.030167,-118.800167,9.59,1.87,md,"27km SE of Bodie, CA",reviewed,...,earthquake,2021,1,30,23,Saturday,CA,US,Mammoth Lakes,North America
2,pr2021030122,2021-01-30 23:39:30.820,1671209101825,17.951600,-66.985300,9.00,1.65,md,"6 km ESE of La Parguera, Puerto Rico",reviewed,...,earthquake,2021,1,30,23,Saturday,Puerto Rico,PR,La Parguera,North America
3,ak0211e015ln,2021-01-30 23:36:27.503,1613163018719,59.183200,-136.452400,7.60,1.70,ml,"35 km SW of Covenant Life, Alaska",reviewed,...,earthquake,2021,1,30,23,Saturday,Alaska,US,Haines,North America
4,ak0211dzz7sr,2021-01-30 23:27:23.670,1613163018483,61.024800,-150.391900,16.60,1.40,ml,"19 km NE of Point Possession, Alaska",reviewed,...,earthquake,2021,1,30,23,Saturday,Alaska,US,Anchorage,North America


In [2]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite://///content/drive/MyDrive/global-seismic-trends/database/earthquakes.db"
)

In [53]:
df.to_sql("earthquakes", engine, if_exists="replace", index=False)

849573

In [54]:
impact_df = pd.read_csv(
    "/content/drive/MyDrive/global-seismic-trends/data/earthquake_impacts_enriched.csv"
)

impact_df.to_sql(
    "earthquake_impacts",
    engine,
    if_exists="replace",
    index=False
)

4827

In [55]:
import os

# os.listdir("/content/drive/MyDrive/global-seismic-trends/database")

pd.read_sql("SELECT (empirical_fatalities+semi_empirical_fatalities) as tot FROM earthquake_impacts ORDER BY tot DESC LIMIT 5", engine)


,tot
0,74667
1,67146
2,26586
3,21573
4,20445


In [56]:
pd.read_sql("SELECT COUNT(*) as tot_row FROM earthquakes", engine)

,tot_row
0,849573


In [57]:
# Q1: Top 10 strongest earthquakes (mag).
q1_query = """
SELECT mag, place, region, time
FROM earthquakes
ORDER BY mag DESC
LIMIT 10
"""

pd.read_sql(q1_query, engine)

,mag,place,region,time
0,8.8,"2025 Kamchatka Peninsula, Russia Earthquake",Russia Earthquake,2025-07-29 23:24:52.483
1,8.2,"2021 Chignik, Alaska Earthquake",Alaska Earthquake,2021-07-29 06:15:49.188
2,8.1,"2021 Kermadec Islands, New Zealand Earthquake",New Zealand Earthquake,2021-03-04 19:28:33.178
3,8.1,2021 South Sandwich Islands Earthquake,2021 South Sandwich Islands Earthquake,2021-08-12 18:35:17.231
4,7.8,"Pazarcik earthquake, Kahramanmaras earthquake ...",Kahramanmaras earthquake sequence,2023-02-06 01:17:34.342
5,7.8,"140 km E of Petropavlovsk-Kamchatsky, Russia",Russia,2025-09-18 18:58:14.939
6,7.8,"25 km SW of Kablalan, Philippines",Philippines,2026-06-07 23:37:41.970
7,7.8,"66 km NNW of Ende, Indonesia",Indonesia,2026-08-14 21:58:21.561
8,7.7,southeast of the Loyalty Islands,southeast of the Loyalty Islands,2021-02-10 13:19:55.530
9,7.7,southeast of the Loyalty Islands,southeast of the Loyalty Islands,2023-05-19 02:57:03.172


In [58]:
#Q2: Top 10 deepest earthquakes (depth_km).
q2_query = """
SELECT depth_km, place, region, time
FROM earthquakes
ORDER BY depth_km DESC
LIMIT 10
"""

pd.read_sql(q2_query, engine)

,depth_km,place,region,time
0,683.578,"206 km ENE of Sola, Vanuatu",Vanuatu,2026-04-02 16:59:17.870
1,681.238,"208 km ENE of Sola, Vanuatu",Vanuatu,2023-04-01 18:09:17.114
2,675.265,Vanuatu region,Vanuatu region,2023-09-18 15:35:27.270
3,671.043,Fiji region,Fiji region,2024-03-08 22:42:23.713
4,669.556,"205 km ESE of Levuka, Fiji",Fiji,2025-10-29 17:07:34.157
5,669.460,"283 km SE of Levuka, Fiji",Fiji,2021-08-01 22:12:46.340
6,667.237,"299 km E of Levuka, Fiji",Fiji,2025-05-10 02:03:16.756
7,667.197,"205 km ESE of Levuka, Fiji",Fiji,2026-02-14 08:27:48.418
8,665.993,"274 km SE of Levuka, Fiji",Fiji,2026-07-05 14:22:25.070
9,665.326,"178 km NE of Sola, Vanuatu",Vanuatu,2026-08-10 16:50:09.877


In [59]:
#Q3: Shallow earthquakes < 50 km and mag > 7.5.
q3_query = """
SELECT depth_km, mag, place, region, time
FROM earthquakes
WHERE depth_km < 50 AND mag > 7.5
"""

pd.read_sql(q3_query, engine)


,depth_km,mag,place,region,time
0,10.000,7.7,southeast of the Loyalty Islands,southeast of the Loyalty Islands,2021-02-10 13:19:55.530
1,28.930,8.1,"2021 Kermadec Islands, New Zealand Earthquake",New Zealand Earthquake,2021-03-04 19:28:33.178
2,35.000,8.2,"2021 Chignik, Alaska Earthquake",Alaska Earthquake,2021-07-29 06:15:49.188
3,22.790,8.1,2021 South Sandwich Islands Earthquake,2021 South Sandwich Islands Earthquake,2021-08-12 18:35:17.231
4,26.943,7.6,"35 km SSW of Aguililla, Mexico",Mexico,2022-09-19 18:05:08.217
5,10.000,7.8,"Pazarcik earthquake, Kahramanmaras earthquake ...",Kahramanmaras earthquake sequence,2023-02-06 01:17:34.342
6,18.053,7.7,southeast of the Loyalty Islands,southeast of the Loyalty Islands,2023-05-19 02:57:03.172
7,40.000,7.6,"19 km E of Gamut, Philippines",Philippines,2023-12-02 14:37:04.454
8,14.326,7.6,"210 km SSW of George Town, Cayman Islands",Cayman Islands,2025-02-08 23:23:14.697
9,10.000,7.7,"2025 Mandalay, Burma (Myanmar) Earthquake",Burma (Myanmar) Earthquake,2025-03-28 06:20:52.715


In [4]:
#Q4:  Average depth per continent
Q4_query = """
SELECT continent, AVG(depth_km) as avg_depth
FROM earthquakes
GROUP BY continent
"""
pd.read_sql(Q4_query, engine)

OperationalError: (sqlite3.OperationalError) unable to open database file
(Background on this error at: https://sqlalche.me/e/21/e3q8)

In [61]:
#Q5: Average magnitude per magnitude type (magType)
q5_query = """
SELECT magType, AVG(mag) as avg_mag
FROM earthquakes
GROUP BY magType
"""

pd.read_sql(q5_query, engine)

,magType,avg_mag
0,None,NaN
1,mb,4.424573
2,mb_lg,2.611813
3,md,1.266335
4,mh,0.337800
5,ml,1.323690
6,ml(texnet),1.987584
7,mlg,3.300000
8,mlr,3.468889
9,mlv,0.911169


In [62]:
#Q6: Year with most earthquakes
q6_query = """
SELECT strftime('%Y', time) as year, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY year
ORDER BY tot_earthquakes DESC
LIMIT 5
"""

pd.read_sql(q6_query, engine)

,year,tot_earthquakes
0,2021,158142
1,2024,150510
2,2023,146515
3,2022,145460
4,2025,135695


In [63]:
#Q7: Month with highest number of earthquakes
q7_query = """
SELECT strftime('%m', time) as month, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY month
ORDER BY tot_earthquakes DESC
LIMIT 5
"""

pd.read_sql(q7_query, engine)

,month,tot_earthquakes
0,07,82358
1,08,78164
2,03,77608
3,01,74917
4,04,73466


In [64]:
#Q8:  Day of week with most earthquakes
q8_query = """
SELECT strftime('%w', time) as day_of_week, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY day_of_week
ORDER BY tot_earthquakes DESC
LIMIT 5"""

pd.read_sql(q8_query, engine)

,day_of_week,tot_earthquakes
0,5,122666
1,4,122263
2,1,122235
3,3,121919
4,2,120374


In [65]:
#Q9: Count of earthquakes per hour of day.
q9_query = """
SELECT strftime('%H', time) as hour_of_day, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY hour_of_day
ORDER BY tot_earthquakes DESC
LIMIT 5"""

pd.read_sql(q9_query, engine)

,hour_of_day,tot_earthquakes
0,08,37101
1,07,36963
2,09,36915
3,10,36512
4,06,36466


In [66]:
#Q10: Most active reporting network (net).
q10_query = """
SELECT net, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY net
ORDER BY tot_earthquakes DESC
LIMIT 5"""

pd.read_sql(q10_query, engine)

,net,tot_earthquakes
0,ak,202808
1,nc,130011
2,us,123128
3,ci,101180
4,av,60762


In [67]:
#Q11: Top 5 places with highest casualties
q11_query = """
SELECT
  e.place, e.region,
  (i.empirical_fatalities+i.semi_empirical_fatalities) as total_casualties
FROM earthquakes e
LEFT JOIN earthquake_impacts i
  ON e.id = i.event_id ORDER BY total_casualties DESC LIMIT 5"""

pd.read_sql(q11_query, engine)

,place,region,total_casualties
0,"20 km W of Catia La Mar, Venezuela",Venezuela,74667
1,"2025 Mandalay, Burma (Myanmar) Earthquake",Burma (Myanmar) Earthquake,67146
2,"Nippes, Haiti",Haiti,26586
3,"Pazarcik earthquake, Kahramanmaras earthquake ...",Kahramanmaras earthquake sequence,21573
4,"20 km E of San Felipe, Venezuela",Venezuela,20445


In [83]:
#Q12: Total estimated economic loss per continent.
q12_query = """SELECT
    e.continent,
    printf('$%,.0f', ROUND(SUM(i.economic_loss_usd), 0)) AS total_economic_loss
FROM earthquakes e
JOIN earthquake_impacts i
    ON e.id = i.event_id
GROUP BY e.continent
ORDER BY SUM(i.economic_loss_usd) DESC"""

pd.read_sql(q12_query, engine)


,continent,total_economic_loss
0,Asia,"$147,363,906,566"
1,South America,"$77,195,875,150"
2,Europe,"$14,077,347,378"
3,North America,"$5,087,548,263"
4,Africa,"$2,537,787,682"
5,Oceania,"$97,862,115"
6,Unknown,"$525,134"


In [84]:
#Q13: Average economic loss by alert level.
q13_query = """
SELECT
    e.alert,
    printf('$%,.0f', ROUND(AVG(i.economic_loss_usd), 0)) AS total_economic_loss
FROM earthquakes e
JOIN earthquake_impacts i
    ON e.id = i.event_id
GROUP BY e.alert
ORDER BY AVG(i.economic_loss_usd) DESC"""

pd.read_sql(q13_query, engine)

,alert,total_economic_loss
0,red,"$8,768,504,360"
1,orange,"$265,483,933"
2,yellow,"$15,244,492"
3,green,"$6,092"


In [70]:
#Q14: Count of reviewed vs automatic earthquakes (status)
q14_query = """
SELECT status, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY status"""

pd.read_sql(q14_query, engine)

,status,tot_earthquakes
0,automatic,97385
1,reviewed,752188


In [71]:
#Q15:  Count by earthquake type (type)
q15_query = """
SELECT type, COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY type"""

pd.read_sql(q15_query, engine)


,type,tot_earthquakes
0,Ice Quake,1
1,Landslide,2
2,accidental explosion,1
3,chemical explosion,14
4,earthquake,829683
5,experimental explosion,7
6,explosion,5621
7,ice quake,4449
8,landslide,38
9,meteorite,2


In [85]:
#Q16: Number of earthquakes by data type (types).
q16_query = """
SELECT
    types,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY types
ORDER BY total_earthquakes DESC
LIMIT 20
"""

pd.read_sql(q16_query, engine)

,types,total_earthquakes
0,",origin,phase-data,",571029
1,",nearby-cities,origin,phase-data,scitech-link,",142188
2,",nearby-cities,origin,phase-data,",44628
3,",focal-mechanism,nearby-cities,origin,phase-da...",28343
4,",dyfi,origin,phase-data,",20438
5,",earthquake-name,origin,phase-data,",7329
6,",dyfi,focal-mechanism,nearby-cities,origin,pha...",6629
7,",dyfi,nearby-cities,origin,phase-data,scitech-...",5333
8,",origin,",3207
9,",origin,phase-data,shakemap,",2750


In [87]:
#Q17: Average RMS and gap per continent.
q17_query = """
SELECT
    continent,
    ROUND(AVG(rms), 2) AS avg_rms,
    ROUND(AVG(gap), 2) AS avg_gap
FROM earthquakes
GROUP BY continent
ORDER BY avg_rms, avg_gap DESC"""

pd.read_sql(q17_query, engine)

,continent,avg_rms,avg_gap
0,North America,0.27,117.84
1,Europe,0.61,155.82
2,Africa,0.67,93.19
3,Oceania,0.70,115.53
4,Unknown,0.70,83.88
5,Asia,0.71,99.68
6,South America,0.72,100.43


In [74]:
#Q18: Events with high station coverage (nst > threshold).
q18_query = """
SELECT
    place,
    region,
    nst
FROM earthquakes
WHERE nst > 500
ORDER BY nst DESC;
"""

pd.read_sql(q18_query, engine)

,place,region,nst
0,"11 km W of Anamizu, Japan",Japan,619.0
1,"120 km ENE of Ozernovskiy, Russia",Russia,566.0
2,"111 km N of Yakutat, Alaska",Alaska,516.0


In [75]:
#Q19:  Number of tsunamis triggered per year
q19_query = """
SELECT
    year,
    COUNT(*) as tot_tsunamis
FROM earthquakes
WHERE tsunami > 0
GROUP BY year
ORDER BY tot_tsunamis DESC
"""

pd.read_sql(q19_query, engine)

,year,tot_tsunamis
0,2025,139
1,2022,131
2,2023,116
3,2024,112
4,2021,109
5,2026,51


In [76]:
#Q20: Count earthquakes by alert levels (red, orange, etc.).
q20_query = """
SELECT
    alert,
    COUNT(*) as tot_earthquakes
FROM earthquakes
GROUP BY alert
ORDER BY tot_earthquakes DESC
"""

pd.read_sql(q20_query, engine)

,alert,tot_earthquakes
0,None,844736
1,green,4641
2,yellow,141
3,orange,28
4,red,27


In [88]:
#Q21: Find the top 5 countries with the highest average magnitude of earthquakes in the past 5 years
q21_query = """
SELECT
    country,
    ROUND(AVG(mag), 2) AS avg_magnitude,
    COUNT(*) AS total_earthquakes
FROM earthquakes
WHERE year >= (
    SELECT MAX(year) - 4
    FROM earthquakes
)
GROUP BY country
HAVING COUNT(*) >= 5
ORDER BY avg_magnitude DESC
LIMIT 5"""

pd.read_sql(q21_query, engine)

,country,avg_magnitude,total_earthquakes
0,Kablalan,5.37,6
1,Tarauaca,5.29,9
2,Aguililla,5.22,8
3,Port Saint John's,5.13,10
4,Satawal,5.12,5


In [95]:
#Q22: Find countries that have experienced both shallow and deep earthquakes within the same month.
q22_query = """
SELECT
    country,
    year,
    month,
    SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END) AS shallow_count,
    SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) AS deep_count
FROM earthquakes
GROUP BY country, year, month
HAVING
    SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END) > 0
    AND
    SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) > 0
ORDER BY year, month, country"""

pd.read_sql(q22_query, engine)

,country,year,month,shallow_count,deep_count
0,Alo,2021,1,3,21
1,Buka,2021,1,3,1
2,Congkar,2021,1,1,2
3,Hihifo,2021,1,7,1
4,Kuril'sk,2021,1,2,1
...,...,...,...,...,...
614,Pangai,2026,9,3,2
615,Saipan,2026,9,7,2
616,`Ohonua,2026,9,16,2
617,Saipan,2026,10,6,2


In [96]:
# df.columns

In [97]:
#Q23: Compute the year-over-year growth rate in the total number of earthquakes globally.
q23_query = pd.read_sql("""
SELECT
    year,
    COUNT(*) AS total_earthquakes
FROM earthquakes
GROUP BY year
ORDER BY year
""", engine)

q23_query["growth_rate_percent"] = (
    q23_query["total_earthquakes"]
    .pct_change() * 100
).round(2)

q23_query

,year,total_earthquakes,growth_rate_percent
0,2021,158142,NaN
1,2022,145460,-8.02
2,2023,146515,0.73
3,2024,150510,2.73
4,2025,135695,-9.84
5,2026,113251,-16.54


In [98]:
#Q24: List the 3 most seismically active regions by combining both frequency and average magnitude.
q24_query = """
SELECT
    region,
    COUNT(*) AS earthquake_count,
    ROUND(AVG(mag), 2) AS avg_magnitude,
    ROUND(
        COUNT(*) * AVG(mag),
        2
    ) AS activity_score
FROM earthquakes
GROUP BY region
HAVING COUNT(*) >= 10
ORDER BY activity_score DESC
LIMIT 3"""

pd.read_sql(q24_query, engine)

,region,earthquake_count,avg_magnitude,activity_score
0,Alaska,279595,1.44,401294.43
1,CA,227333,1.04,236798.24
2,Hawaii,45709,1.89,86436.87


In [99]:
#Q25: For each country, calculate the average depth of earthquakes within ±5° latitude range of the equator.
q25_query = """SELECT
    country,
    ROUND(AVG(depth_km), 2) AS avg_depth_km,
    COUNT(*) AS earthquake_count
FROM earthquakes
WHERE latitude BETWEEN -5 AND 5
GROUP BY country
ORDER BY avg_depth_km DESC"""

pd.read_sql(q25_query, engine)

,country,avg_depth_km,earthquake_count
0,Latung,600.00,3
1,Palimbang,564.77,3
2,Maitum,513.10,3
3,Malisbeng,506.87,3
4,Tabiauan,501.04,26
...,...,...,...
635,Otavalo,7.90,1
636,Kongolo,7.89,1
637,Blang Ara,7.78,2
638,San Gabriel,6.80,4


In [100]:
#Q26: Identify countries having the highest ratio of shallow to deep earthquakes.
q26_query = """
SELECT
    country,
    SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END) AS shallow_count,
    SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END) AS deep_count,
    ROUND(
        CAST(
            SUM(CASE WHEN depth_km < 70 THEN 1 ELSE 0 END)
            AS REAL
        ) /
        NULLIF(
            SUM(CASE WHEN depth_km > 300 THEN 1 ELSE 0 END),
            0
        ),
        2
    ) AS shallow_to_deep_ratio
FROM earthquakes
GROUP BY country
HAVING deep_count > 0
ORDER BY shallow_to_deep_ratio DESC"""

pd.read_sql(q26_query, engine)


,country,shallow_count,deep_count,shallow_to_deep_ratio
0,Dillingham,5081,1,5081.0
1,Trinidad,767,1,767.0
2,Isangel,420,1,420.0
3,Sarangani,260,1,260.0
4,Nanao,100,1,100.0
...,...,...,...,...
202,Anacapri,0,2,0.0
203,Aloindonu,0,7,0.0
204,Alemba,0,14,0.0
205,Alamada,0,1,0.0


In [102]:
#Q27: Find the average magnitude difference between earthquakes with tsunami alerts and those without.
q27_query = """
SELECT
    ROUND(
        (
            SELECT AVG(mag)
            FROM earthquakes
            WHERE tsunami = 1
        )
        -
        (
            SELECT AVG(mag)
            FROM earthquakes
            WHERE tsunami = 0
        ),
        2
    ) AS avg_magnitude_difference"""

pd.read_sql(q27_query, engine)

,avg_magnitude_difference
0,3.77


In [103]:
#Q28: Using the gap and rms columns, identify events with the lowest data reliability (highest average error margins).
q28_query = """
SELECT
    id,
    place,
    country,
    mag,
    rms,
    gap,
    ROUND((rms + gap) / 2.0, 2) AS avg_error_margin
FROM earthquakes
ORDER BY avg_error_margin DESC
LIMIT 20"""

pd.read_sql(q28_query, engine)

,id,place,country,mag,rms,gap,avg_error_margin
0,uw61988777,"9 km N of Peaceful Valley, Washington",Peaceful Valley,1.06,0.5000,360.0,180.25
1,ci40440272,"8km SE of Avalon, CA",Avalon,1.20,0.2200,360.0,180.11
2,ci39657847,"8km SE of Avalon, CA",Avalon,1.44,0.2100,360.0,180.10
3,ci41277408,"10 km NNW of Thousand Palms, CA",Desert Edge,0.05,0.2000,360.0,180.10
4,ci39623194,"8km SE of Avalon, CA",Avalon,1.17,0.1900,360.0,180.09
5,nn00907449,"59 km NE of Valmy, Nevada",Battle Mountain,2.13,0.1436,360.0,180.07
6,uw61718587,"22 km NNW of Benton City, Washington",Benton City,-0.18,0.1200,360.0,180.06
7,ci39902544,"18km E of Ridgecrest, CA",Searles Valley,-0.24,0.1300,360.0,180.06
8,nc73578255,"12km SE of Pinnacles, CA",Greenfield,0.59,0.0900,360.0,180.04
9,uw61950867,Washington,West Pasco,0.53,0.0800,360.0,180.04


In [104]:
#Q29: Find pairs of consecutive earthquakes (by time) that occurred within 50 km of each other and within 1 hour.
#To-do

In [106]:
#Q30: Determine the regions with the highest frequency of deep-focus earthquakes (depth > 300 km).
q30_query = """
SELECT
    region,
    COUNT(*) AS deep_focus_earthquake_count
FROM earthquakes
WHERE depth_km > 300
GROUP BY region
ORDER BY deep_focus_earthquake_count DESC"""

pd.read_sql(q30_query, engine)

,region,deep_focus_earthquake_count
0,south of the Fiji Islands,1697
1,Fiji region,1274
2,Fiji,1203
3,Tonga,615
4,Indonesia,286
5,Japan region,248
6,Timor Leste,219
7,Kermadec Islands region,175
8,Wallis and Futuna,167
9,Northern Mariana Islands,132
